# Notebook 01: Environment Setup and External API Verification
**GlobalFlood AI: Global Multimodal Flood Prediction System (48-Hour Forecast)**

### Objective
Verify GPU hardware acceleration (CUDA / Apple Silicon MPS / CPU), install necessary Python dependencies, and test live connectivity to all four required open data APIs:
1. Open-Meteo High-Resolution Forecast API
2. Open-Meteo Historical Archive API (ERA5 reanalysis)
3. Open-Meteo Global Elevation API
4. Copernicus Data Space Ecosystem (CDSE) Sentinel-1 OData Catalog

### Inputs & Outputs
- **Inputs:** Coordinates for reference site (Chennai coastal basin: 13.0827°N, 80.2707°E)
- **Outputs:** Verification report confirming connectivity, API latency, and GPU availability
- **Dependencies:** `torch`, `httpx`, `pydantic`


In [ ]:
import sys
import torch
import httpx

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
cuda_avail = torch.cuda.is_available()
mps_avail = torch.backends.mps.is_available()
print(f"CUDA Available: {cuda_avail}")
print(f"Apple Silicon MPS Available: {mps_avail}")

if cuda_avail:
    device = torch.device("cuda")
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
elif mps_avail:
    device = torch.device("mps")
    print("Active Acceleration: Apple Silicon Metal Performance Shaders (MPS)")
else:
    device = torch.device("cpu")
    print("Active Device: CPU")


### Test 1: Open-Meteo Forecast API
Tests retrieval of hourly precipitation, temperature, humidity, pressure, wind speed, and soil moisture.


In [ ]:
lat, lon = 13.0827, 80.2707
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": lat,
    "longitude": lon,
    "hourly": ["precipitation", "temperature_2m", "relative_humidity_2m", "surface_pressure", "wind_speed_10m", "soil_moisture_0_to_1cm"],
    "past_days": 2,
    "forecast_days": 2,
    "timezone": "UTC"
}
with httpx.Client(timeout=15.0) as client:
    resp = client.get(url, params=params)
    print(f"Open-Meteo Forecast Status: {resp.status_code}")
    assert resp.status_code == 200
    data = resp.json()
    print("Variables returned:", list(data.get("hourly", {}).keys()))
    print(f"Total time steps retrieved: {len(data['hourly']['time'])}")


### Test 2: Open-Meteo Historical Reanalysis API
Tests retrieval of ERA5 historical weather reanalysis for a past date window.


In [ ]:
hist_url = "https://archive-api.open-meteo.com/v1/archive"
hist_params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2020-10-01",
    "end_date": "2020-10-07",
    "hourly": ["precipitation", "temperature_2m", "relative_humidity_2m", "surface_pressure"],
    "timezone": "UTC"
}
with httpx.Client(timeout=15.0) as client:
    resp = client.get(hist_url, params=hist_params)
    print(f"Open-Meteo Archive Status: {resp.status_code}")
    assert resp.status_code == 200
    print(f"Historical hours retrieved: {len(resp.json()['hourly']['time'])}")


### Test 3: Open-Meteo Global Elevation API
Queries 90m Copernicus DEM / global elevation.


In [ ]:
elev_url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
with httpx.Client(timeout=10.0) as client:
    resp = client.get(elev_url)
    print(f"Elevation Status: {resp.status_code}")
    assert resp.status_code == 200
    elevation = resp.json().get("elevation", [None])[0]
    print(f"Elevation at ({lat}, {lon}): {elevation} meters ASL")


### Test 4: Copernicus Data Space Ecosystem (CDSE) Sentinel-1 Catalog
Queries the European Space Agency (ESA) CDSE OData product catalog for Sentinel-1 GRD SAR products.


In [ ]:
cdse_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
filter_q = "Collection/Name eq 'SENTINEL-1' and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'GRD')"
params = {"$top": 2, "$filter": filter_q}
try:
    with httpx.Client(timeout=15.0) as client:
        resp = client.get(cdse_url, params=params)
        print(f"CDSE OData Status: {resp.status_code}")
        if resp.status_code == 200:
            val = resp.json().get("value", [])
            print(f"Found {len(val)} products. Example product ID: {val[0].get('Name') if val else 'None'}")
        else:
            print("CDSE response code:", resp.status_code)
except Exception as e:
    print(f"CDSE query note: {e}")

print("\n[SUCCESS] Environment and API connectivity test completed successfully.")
